# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library. All dataset structures (record sets, fields, columns) are referenced by their `@id` for accuracy and reproducibility.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, their IDs, and get an overview of the dataset structure.

In [ ]:
# List all available record sets by their @id
print("Available record sets (by @id):")
for record_set in metadata.record_sets:
    print(f"- {record_set['@id']}: {record_set.get('name', '')}")

# For each record set, list its fields (by @id) and columns
record_sets_overview = {}
for record_set in metadata.record_sets:
    rec_id = record_set['@id']
    fields = record_set.get('fields', [])
    columns = record_set.get('columns', [])
    print(f"\nRecord set '{rec_id}':")
    print("  Fields (by @id):")
    for f in fields:
        print(f"    {f['@id']}: {f.get('name', '')} ({f.get('dataType', '')})")
    print("  Columns (by @id):")
    for c in columns:
        print(f"    {c['@id']}: {c.get('name', '')}")
    record_sets_overview[rec_id] = {
        'fields': [f['@id'] for f in fields],
        'columns': [c['@id'] for c in columns]
    }


## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Refer to the record set and field `@id`s from the overview above. If multiple record sets are present, all are extracted.

In [ ]:
# Get all record set @id's
record_set_ids = [r['@id'] for r in metadata.record_sets]
dataframes = {}
for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded record set '{record_set_id}' with shape {df.shape}")
        print(f"Columns: {df.columns.tolist()}")
    except Exception as e:
        print(f"Could not load records from '{record_set_id}': {e}")

# Preview the first DataFrame
if record_set_ids:
    print(f"\nPreview of '{record_set_ids[0]}' (first 5 rows):")
    display(dataframes[record_set_ids[0]].head())

## 4. Exploratory Data Analysis (EDA)
Apply simple transformation and filtering using the fields `@id`.

Choose a numeric field for demonstration. You can adjust the field name below based on the printed columns in the previous cell.

In [ ]:
# Example: Select the first record set and a numeric field for analysis
import numpy as np

# Replace these according to your dataset structure
analyzed_record_set = record_set_ids[0] if record_set_ids else None

# Select a numeric field (by @id/column name). Adjust as appropriate.
numeric_candidates = [c for c in dataframes[analyzed_record_set].columns if dataframes[analyzed_record_set][c].dtype in [np.float64, np.int64, float, int]]
if numeric_candidates:
    numeric_field_id = numeric_candidates[0]
    print(f"Using numeric field @id: {numeric_field_id}")
else:
    # Pick the first column for demo, if no numeric detected
    numeric_field_id = dataframes[analyzed_record_set].columns[0]
    print(f"No numeric field detected! Defaulting to: {numeric_field_id}")

# Remove missing values for this field, filter for > threshold (mean for demo)
df = dataframes[analyzed_record_set].copy()
df = df[pd.to_numeric(df[numeric_field_id], errors='coerce').notna()].copy()
df[numeric_field_id] = df[numeric_field_id].astype(float)
threshold = df[numeric_field_id].mean()
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records in '{analyzed_record_set}' with {numeric_field_id} > mean ({threshold:.2f}):")
display(filtered_df.head())

# Normalize the selected numeric field (z-score)
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized '{numeric_field_id}' for filtered records:")
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Try grouping by another field (choose a non-numeric if available)
group_field_candidates = [col for col in df.columns if col != numeric_field_id and not np.issubdtype(df[col].dtype, np.number)]
if group_field_candidates:
    group_field = group_field_candidates[0]
    print(f"Grouping by field @id: {group_field}\n")
    grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().to_frame()
    print(f"Grouped mean of '{numeric_field_id}' by '{group_field}':")
    display(grouped_df.head())

## 5. Visualization
Visualize the distribution of the selected numeric field and group averages if applicable.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram of the numeric field
plt.figure(figsize=(7,4))
sns.histplot(df[numeric_field_id], kde=True, bins=20, color="royalblue")
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.show()

# If grouped data is available, plot group averages
if 'grouped_df' in locals():
    grouped_df.plot(kind='bar', legend=False, figsize=(8,4))
    plt.title(f"Mean {numeric_field_id} by {group_field}")
    plt.xlabel(group_field)
    plt.ylabel(f"Mean {numeric_field_id}")
    plt.tight_layout()
    plt.show()

## 6. Conclusion
In this notebook, we used the `mlcroissant` library to load the FAIR^2 dataset via its Croissant schema, explored the key record sets, and demonstrated common EDA and visualization steps using field and record set `@id` references. Adjust analysis steps and field selection as per your research questions. For more details about dataset semantics or for advanced transformations, refer to the dataset's schema documentation at [sen.science](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json).